<a href="https://colab.research.google.com/github/CassieMarie0728/colab-notebooks/blob/main/RVC_Dataset_Preprocessor_Colab_UPDATED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RVC Dataset Preprocessor — Colab Notebook

This notebook builds a clean, organized, RVC-style dataset from raw vocal files.

**Updated for Cass's 14-part RVC monologue set**
- accepts either loose audio files **or a ZIP containing audio**
- keeps one clean dataset run per notebook execution
- uses less aggressive slicing so useful short phrases do not get tossed like trash
- merges short speech chunks instead of rejecting half the damn script
- reports total kept audio time, rejected audio time, and keep ratio
- builds `raw/`, `sliced/`, `logs/0_gt_wavs/`, `filelist.txt`, CSV reports, and a downloadable ZIP

## What it does
- Upload source audio or a dataset ZIP
- Convert to mono WAV
- Resample to your target sample rate
- Normalize peak safely
- Optionally run light denoise
- Slice audio into training-friendly chunks
- Merge short chunks before rejecting them
- Reject clips that are truly too short or too quiet
- Build an RVC-style folder layout
- Generate `filelist.txt`
- Export the dataset as a ZIP

## What it does not do
- HuBERT feature extraction
- F0 extraction
- RVC training
- stem separation

This is for dataset prep, not the full training pipeline.


## 1) Install dependencies

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install librosa soundfile pydub tqdm noisereduce numpy scipy pandas matplotlib

## 2) Imports

In [ ]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from pydub import AudioSegment, silence
from google.colab import files

warnings.filterwarnings("ignore")

## 3) Configuration

These defaults are tuned for your cleaned spoken-word RVC dataset.

The important changes from the old notebook:
- **less aggressive silence detection**
- **short chunks get merged before rejection**
- **minimum clip length is lowered**
- **peak normalization target is safer**

Recommended defaults for this dataset:
- `TARGET_SR = 40000`
- `MIN_CLIP_MS = 1500`
- `MAX_CLIP_MS = 15000`
- `MIN_SILENCE_LEN_MS = 650`
- `SILENCE_THRESH_DBFS = -50`
- `KEEP_SILENCE_MS = 300`

If it still rejects too much:
- lower `MIN_CLIP_MS` to `1200`
- lower `SILENCE_THRESH_DBFS` to `-55`
- increase `KEEP_SILENCE_MS` to `400`

If clips are too long or barely sliced:
- raise `SILENCE_THRESH_DBFS` to `-45`
- lower `MIN_SILENCE_LEN_MS` to `500`


In [ ]:
DATASET_NAME = "cass_rvc_dataset_clean"
SPEAKER_ID = 0

# RVC commonly supports 40000 / 48000. Keep this at 40000 if your training config expects 40k.
TARGET_SR = 40000
MONO = True

# Peak-normalize lower than before so loud consonants do not scrape the ceiling.
NORMALIZE_AUDIO = True
PEAK_TARGET = 0.90

USE_DENOISE = False
TRIM_LEADING_TRAILING_SILENCE = True
TRIM_TOP_DB = 40

# Less aggressive slicing for spoken monologue data.
MIN_SILENCE_LEN_MS = 650
SILENCE_THRESH_DBFS = -50
KEEP_SILENCE_MS = 300

# Lower minimum + longer maximum = fewer useful phrases rejected.
MIN_CLIP_MS = 1500
MAX_CLIP_MS = 15000
MIN_RMS = 0.003

# If True, every run starts clean. Keeps old files from sneaking into the new dataset like goblins.
CLEAR_PREVIOUS_OUTPUT = True

AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".opus"}
ARCHIVE_EXTS = {".zip"}
ALLOWED_EXTS = AUDIO_EXTS | ARCHIVE_EXTS

BASE_DIR = Path("/content") / DATASET_NAME
RAW_DIR = BASE_DIR / "raw"
SLICED_DIR = BASE_DIR / "sliced"
LOGS_DIR = BASE_DIR / "logs"
GT_WAVS_DIR = LOGS_DIR / "0_gt_wavs"
REPORTS_DIR = BASE_DIR / "reports"

if CLEAR_PREVIOUS_OUTPUT and BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)

for d in [BASE_DIR, RAW_DIR, SLICED_DIR, LOGS_DIR, GT_WAVS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Dataset root: {BASE_DIR}")
print("Slicing defaults:")
print(f"  MIN_CLIP_MS={MIN_CLIP_MS}, MAX_CLIP_MS={MAX_CLIP_MS}")
print(f"  MIN_SILENCE_LEN_MS={MIN_SILENCE_LEN_MS}, SILENCE_THRESH_DBFS={SILENCE_THRESH_DBFS}, KEEP_SILENCE_MS={KEEP_SILENCE_MS}")


## 4) Upload your source audio

You can upload:
- the final dataset ZIP, or
- the individual WAV files

If you upload a ZIP, this cell extracts audio files from it and ignores junk like folders, hidden files, and non-audio files.


In [ ]:
import zipfile

uploaded = files.upload()

print(f"Uploaded {len(uploaded)} file(s).")

saved_files = []

def unique_path(directory: Path, filename: str) -> Path:
    filename = Path(filename).name
    stem = Path(filename).stem
    suffix = Path(filename).suffix
    candidate = directory / filename
    counter = 1
    while candidate.exists():
        candidate = directory / f"{stem}_{counter}{suffix}"
        counter += 1
    return candidate

for name, data in uploaded.items():
    ext = Path(name).suffix.lower()

    if ext in ARCHIVE_EXTS:
        archive_path = RAW_DIR / Path(name).name
        with open(archive_path, "wb") as f:
            f.write(data)

        extract_dir = RAW_DIR / f"extracted_{safe_stem(name)}"
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(archive_path, "r") as z:
            z.extractall(extract_dir)

        for p in sorted(extract_dir.rglob("*")):
            if p.is_file() and p.suffix.lower() in AUDIO_EXTS and not p.name.startswith("._"):
                out_path = unique_path(RAW_DIR, p.name)
                shutil.copy2(p, out_path)
                saved_files.append(out_path)

        print(f"Extracted audio from ZIP: {name}")
        continue

    if ext not in AUDIO_EXTS:
        print(f"Skipping unsupported file: {name}")
        continue

    out_path = unique_path(RAW_DIR, name)
    with open(out_path, "wb") as f:
        f.write(data)
    saved_files.append(out_path)

print(f"Saved {len(saved_files)} audio file(s):")
for p in saved_files:
    print(" -", p)


## 5) Inspect uploaded files

In [ ]:
raw_files = sorted([
    p for p in RAW_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in AUDIO_EXTS
])

rows = []
for p in raw_files:
    try:
        info = sf.info(str(p))
        rows.append({
            "file": p.name,
            "samplerate": info.samplerate,
            "channels": info.channels,
            "frames": info.frames,
            "duration_sec": round(info.frames / info.samplerate, 2),
            "format": info.format,
            "subtype": info.subtype,
        })
    except Exception:
        rows.append({
            "file": p.name,
            "samplerate": None,
            "channels": None,
            "frames": None,
            "duration_sec": None,
            "format": "unknown",
            "subtype": "unknown",
        })

df_raw = pd.DataFrame(rows)
print(f"Found {len(df_raw)} source audio file(s).")
print(f"Total source duration: {df_raw['duration_sec'].sum():.2f} sec / {df_raw['duration_sec'].sum()/60:.2f} min" if len(df_raw) else "No audio files found.")
df_raw


## 6) Helper functions

In [ ]:
import noisereduce as nr

def safe_stem(name: str) -> str:
    stem = Path(name).stem
    stem = re.sub(r"[^a-zA-Z0-9_\-]+", "_", stem)
    stem = re.sub(r"_+", "_", stem).strip("_")
    return stem or "audio"

def load_audio_any(path, sr=40000, mono=True):
    y, _ = librosa.load(path, sr=sr, mono=mono)
    return y.astype(np.float32)

def trim_silence(y, top_db=40):
    yt, _ = librosa.effects.trim(y, top_db=top_db)
    return yt.astype(np.float32)

def peak_normalize(y, peak_target=0.90):
    peak = np.max(np.abs(y)) if len(y) else 0
    if peak <= 0:
        return y
    return (y / peak * peak_target).astype(np.float32)

def rms(y):
    if len(y) == 0:
        return 0.0
    return float(np.sqrt(np.mean(np.square(y))))

def denoise_audio(y, sr):
    return nr.reduce_noise(y=y, sr=sr, stationary=True, prop_decrease=0.75).astype(np.float32)

def save_wav(path, y, sr):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), y, sr, subtype="PCM_16")

def wav_to_audiosegment(path):
    return AudioSegment.from_wav(path)

def split_long_segment(seg, max_len_ms):
    if len(seg) <= max_len_ms:
        return [seg]
    pieces = []
    start = 0
    while start < len(seg):
        end = min(start + max_len_ms, len(seg))
        pieces.append(seg[start:end])
        start = end
    return pieces

def merge_short_segments(chunks, min_len_ms, max_len_ms):
    """Merge adjacent short chunks so the slicer does not reject useful short phrases."""
    merged = []
    buffer = AudioSegment.silent(duration=0)

    for chunk in chunks:
        if len(buffer) == 0:
            buffer = chunk
        elif len(buffer) < min_len_ms or len(chunk) < min_len_ms:
            if len(buffer) + len(chunk) <= max_len_ms:
                buffer += chunk
            else:
                merged.append(buffer)
                buffer = chunk
        else:
            merged.append(buffer)
            buffer = chunk

    if len(buffer) > 0:
        merged.append(buffer)

    return merged

def clip_is_valid(y, min_rms=0.003):
    return len(y) > 0 and rms(y) >= min_rms and np.max(np.abs(y)) > 1e-4

def audiosegment_to_float(piece, sr):
    samples = np.array(piece.get_array_of_samples()).astype(np.float32)
    if piece.sample_width == 2:
        samples /= 32768.0
    elif piece.sample_width == 4:
        samples /= 2147483648.0
    else:
        samples /= max(float(1 << (8 * piece.sample_width - 1)), 1.0)
    return samples.astype(np.float32)


## 7) Convert, clean, and normalize source files

In [ ]:
CLEANED_RAW_DIR = RAW_DIR / "cleaned_wavs"
CLEANED_RAW_DIR.mkdir(parents=True, exist_ok=True)

cleaned_rows = []

for src in tqdm(raw_files, desc="Cleaning source audio"):
    try:
        y = load_audio_any(src, sr=TARGET_SR, mono=MONO)

        if TRIM_LEADING_TRAILING_SILENCE:
            y = trim_silence(y, top_db=TRIM_TOP_DB)

        if USE_DENOISE and len(y) > 0:
            y = denoise_audio(y, TARGET_SR)

        if NORMALIZE_AUDIO and len(y) > 0:
            y = peak_normalize(y, peak_target=PEAK_TARGET)

        out_name = f"{safe_stem(src.name)}.wav"
        out_path = CLEANED_RAW_DIR / out_name
        save_wav(out_path, y, TARGET_SR)

        cleaned_rows.append({
            "source_file": src.name,
            "cleaned_file": out_name,
            "duration_sec": round(len(y) / TARGET_SR, 2),
            "rms": round(rms(y), 6),
            "peak": round(float(np.max(np.abs(y))) if len(y) else 0.0, 6),
            "status": "ok"
        })
    except Exception as e:
        cleaned_rows.append({
            "source_file": src.name,
            "cleaned_file": None,
            "duration_sec": None,
            "rms": None,
            "peak": None,
            "status": f"error: {e}"
        })

df_cleaned = pd.DataFrame(cleaned_rows)
print(f"Total cleaned duration: {df_cleaned['duration_sec'].sum():.2f} sec / {df_cleaned['duration_sec'].sum()/60:.2f} min" if len(df_cleaned) else "No cleaned audio.")
df_cleaned


## 8) Slice cleaned audio into training clips

In [ ]:
cleaned_wavs = sorted(CLEANED_RAW_DIR.glob("*.wav"))

slice_rows = []

for cleaned_path in tqdm(cleaned_wavs, desc="Slicing audio"):
    base = safe_stem(cleaned_path.name)

    try:
        seg = wav_to_audiosegment(cleaned_path)
        if seg.channels != 1:
            seg = seg.set_channels(1)
        if seg.frame_rate != TARGET_SR:
            seg = seg.set_frame_rate(TARGET_SR)

        chunks = silence.split_on_silence(
            seg,
            min_silence_len=MIN_SILENCE_LEN_MS,
            silence_thresh=SILENCE_THRESH_DBFS,
            keep_silence=KEEP_SILENCE_MS,
        )

        if not chunks:
            chunks = [seg]

        # Big fix: merge tiny speech chunks before judging them.
        chunks = merge_short_segments(chunks, MIN_CLIP_MS, MAX_CLIP_MS)

        clip_index = 0
        kept = 0
        rejected = 0
        kept_ms = 0
        rejected_ms = 0
        rejected_too_short = 0
        rejected_too_quiet = 0

        for chunk in chunks:
            for piece in split_long_segment(chunk, MAX_CLIP_MS):
                if len(piece) < MIN_CLIP_MS:
                    rejected += 1
                    rejected_too_short += 1
                    rejected_ms += len(piece)
                    continue

                y = audiosegment_to_float(piece, TARGET_SR)

                if not clip_is_valid(y, min_rms=MIN_RMS):
                    rejected += 1
                    rejected_too_quiet += 1
                    rejected_ms += len(piece)
                    continue

                if NORMALIZE_AUDIO:
                    y = peak_normalize(y, peak_target=PEAK_TARGET)

                clip_name = f"{base}_{clip_index:04d}.wav"
                temp_path = SLICED_DIR / clip_name
                gt_path = GT_WAVS_DIR / clip_name

                save_wav(temp_path, y, TARGET_SR)
                shutil.copy2(temp_path, gt_path)

                kept += 1
                kept_ms += len(piece)
                clip_index += 1

        total_ms = max(len(seg), 1)
        slice_rows.append({
            "file": cleaned_path.name,
            "input_sec": round(len(seg) / 1000, 2),
            "clips_kept": kept,
            "clips_rejected": rejected,
            "kept_sec": round(kept_ms / 1000, 2),
            "rejected_sec": round(rejected_ms / 1000, 2),
            "keep_ratio": round(kept_ms / total_ms, 3),
            "rejected_too_short": rejected_too_short,
            "rejected_too_quiet": rejected_too_quiet,
            "status": "ok"
        })

    except Exception as e:
        slice_rows.append({
            "file": cleaned_path.name,
            "input_sec": 0,
            "clips_kept": 0,
            "clips_rejected": 0,
            "kept_sec": 0,
            "rejected_sec": 0,
            "keep_ratio": 0,
            "rejected_too_short": 0,
            "rejected_too_quiet": 0,
            "status": f"error: {e}"
        })

df_slices = pd.DataFrame(slice_rows)

if len(df_slices):
    print(f"Total kept clips: {int(df_slices['clips_kept'].sum())}")
    print(f"Total rejected chunks: {int(df_slices['clips_rejected'].sum())}")
    print(f"Total kept duration: {df_slices['kept_sec'].sum():.2f} sec / {df_slices['kept_sec'].sum()/60:.2f} min")
    print(f"Overall keep ratio: {df_slices['kept_sec'].sum() / max(df_slices['input_sec'].sum(), 1):.1%}")

df_slices


## 9) Review the generated clips

In [ ]:
generated_clips = sorted(GT_WAVS_DIR.glob("*.wav"))

rows = []
for p in generated_clips:
    info = sf.info(str(p))
    y, _ = librosa.load(p, sr=TARGET_SR, mono=True)
    rows.append({
        "clip": p.name,
        "duration_sec": round(info.frames / info.samplerate, 2),
        "rms": round(rms(y), 6),
        "peak": round(float(np.max(np.abs(y))) if len(y) else 0.0, 6),
    })

df_clips = pd.DataFrame(rows)
if len(df_clips):
    df_clips = df_clips.sort_values(["duration_sec", "clip"], ascending=[False, True])
    total_clip_sec = df_clips["duration_sec"].sum()
    print(f"Total generated clips: {len(df_clips)}")
    print(f"Total generated duration: {total_clip_sec:.2f} sec / {total_clip_sec/60:.2f} min")
    print(f"Average clip length: {df_clips['duration_sec'].mean():.2f} sec")
    print(f"Shortest clip: {df_clips['duration_sec'].min():.2f} sec")
    print(f"Longest clip: {df_clips['duration_sec'].max():.2f} sec")

    if total_clip_sec < 600:
        print("⚠️ Under 10 minutes kept. Consider lowering MIN_CLIP_MS or making slicing less aggressive.")
    elif total_clip_sec < 900:
        print("✅ Usable. More would be nice, but this can train.")
    else:
        print("✅ Good dataset length for a first speaking RVC model.")

    display(df_clips.head(25))
else:
    print("No clips found. Adjust your slicing settings and rerun.")
    df_clips


## 10) Quick visualization

In [ ]:
if len(df_clips) > 0:
    plt.figure(figsize=(12, 4))
    plt.hist(df_clips["duration_sec"], bins=25)
    plt.title("Clip Duration Distribution")
    plt.xlabel("Seconds")
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.hist(df_clips["rms"], bins=25)
    plt.title("Clip RMS Distribution")
    plt.xlabel("RMS")
    plt.ylabel("Count")
    plt.show()
else:
    print("No clips found. Adjust your slicing settings and rerun.")

## 11) Build `filelist.txt`

In [ ]:
filelist_path = BASE_DIR / "filelist.txt"

with open(filelist_path, "w", encoding="utf-8") as f:
    for wav_path in sorted(GT_WAVS_DIR.glob("*.wav")):
        rel = wav_path.relative_to(BASE_DIR).as_posix()
        f.write(f"{rel}|{SPEAKER_ID}\n")

print(f"Wrote: {filelist_path}")

with open(filelist_path, "r", encoding="utf-8") as f:
    preview = "".join(f.readlines()[:10])

print("\nPreview:\n")
print(preview if preview else "[filelist is empty]")

## 12) Save reports

In [ ]:
df_raw.to_csv(REPORTS_DIR / "raw_files_report.csv", index=False)
df_cleaned.to_csv(REPORTS_DIR / "cleaned_files_report.csv", index=False)
df_slices.to_csv(REPORTS_DIR / "slice_report.csv", index=False)
df_clips.to_csv(REPORTS_DIR / "generated_clips_report.csv", index=False)

total_source_sec = float(df_raw["duration_sec"].sum()) if len(df_raw) else 0.0
total_cleaned_sec = float(df_cleaned["duration_sec"].sum()) if len(df_cleaned) else 0.0
total_kept_sec = float(df_clips["duration_sec"].sum()) if len(df_clips) else 0.0
overall_keep_ratio = total_kept_sec / total_cleaned_sec if total_cleaned_sec else 0.0

summary = {
    "dataset_name": DATASET_NAME,
    "speaker_id": SPEAKER_ID,
    "target_sr": TARGET_SR,
    "total_source_files": int(len(df_raw)),
    "total_cleaned_files": int((df_cleaned["status"] == "ok").sum()) if len(df_cleaned) else 0,
    "total_generated_clips": int(len(df_clips)),
    "total_source_sec": round(total_source_sec, 2),
    "total_cleaned_sec": round(total_cleaned_sec, 2),
    "total_kept_sec": round(total_kept_sec, 2),
    "total_kept_min": round(total_kept_sec / 60, 2),
    "overall_keep_ratio": round(overall_keep_ratio, 3),
    "settings": {
        "mono": MONO,
        "normalize_audio": NORMALIZE_AUDIO,
        "peak_target": PEAK_TARGET,
        "use_denoise": USE_DENOISE,
        "trim_leading_trailing_silence": TRIM_LEADING_TRAILING_SILENCE,
        "trim_top_db": TRIM_TOP_DB,
        "min_silence_len_ms": MIN_SILENCE_LEN_MS,
        "silence_thresh_dbfs": SILENCE_THRESH_DBFS,
        "keep_silence_ms": KEEP_SILENCE_MS,
        "min_clip_ms": MIN_CLIP_MS,
        "max_clip_ms": MAX_CLIP_MS,
        "min_rms": MIN_RMS,
        "merge_short_segments": True,
    }
}

with open(REPORTS_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved report files to:", REPORTS_DIR)
print(json.dumps(summary, indent=2))


## 13) Package the dataset as a ZIP

In [ ]:
zip_base = f"/content/{DATASET_NAME}"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=BASE_DIR)
print("Created ZIP:", zip_path)

## 14) Download the ZIP

In [ ]:
files.download(f"/content/{DATASET_NAME}.zip")

## Folder layout produced

```text
cass_rvc_dataset_clean/
├── raw/
│   ├── original uploads
│   ├── extracted ZIP contents, if any
│   └── cleaned_wavs/
├── sliced/
├── logs/
│   └── 0_gt_wavs/
├── reports/
│   ├── raw_files_report.csv
│   ├── cleaned_files_report.csv
│   ├── slice_report.csv
│   ├── generated_clips_report.csv
│   └── summary.json
├── filelist.txt
```

## Important reality check

This notebook prepares the dataset structure and training clips.

It does not generate:
- `2a_f0/`
- `2b-f0nsf/`
- `3_feature256/`

Those are created later in the actual RVC preprocessing / feature extraction pipeline.

## If the slicer is still being a picky little goblin

Change these in Configuration and rerun from the top:

```python
MIN_CLIP_MS = 1200
SILENCE_THRESH_DBFS = -55
KEEP_SILENCE_MS = 400
```

If the clips are too long:

```python
SILENCE_THRESH_DBFS = -45
MIN_SILENCE_LEN_MS = 500
```
